# 01 — Data Preparation
**Steel Surface Defect Detection (NEU-DET + YOLOv8)**

This notebook:
1. Downloads the NEU-DET dataset
2. Converts Pascal VOC XML annotations to YOLO `.txt` format
3. Builds the `images/` + `labels/` **train/val/test** structure YOLOv8 expects, using the
   paper's **8:1:1** split (1440 / 180 / 180)

Run the cells top to bottom.

## 1. Imports & paths

In [3]:
import os, shutil, random
from pathlib import Path
import xml.etree.ElementTree as ET

# Project root = parent of the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
print('Project root:', ROOT)
print('Data dir:', DATA)

Project root: c:\Users\student\Desktop\SteelDefectDetection
Data dir: c:\Users\student\Desktop\SteelDefectDetection\data


## 2. Locate NEU-DET
The raw dataset already ships in this repo at `data/NEU-DET/` (Pascal-VOC format), so **no download is needed** — that's the default below.

If you ever want a fresh copy, set `USE_KAGGLEHUB = True`.
Kaggle: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database

In [4]:
USE_KAGGLEHUB = False  # data already lives in data/NEU-DET — no download needed

if USE_KAGGLEHUB:
    import kagglehub
    raw_path = kagglehub.dataset_download('kaustubhdikshit/neu-surface-defect-database')
    RAW_DIR = Path(raw_path)
    print('Downloaded to:', RAW_DIR)
else:
    RAW_DIR = DATA / 'NEU-DET'
    print('Using local data:', RAW_DIR)

assert RAW_DIR.exists(), f'NEU-DET not found at {RAW_DIR}'

Using local data: c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET


## 3. Locate images & annotations
NEU-DET copies vary in layout. This cell searches for the folders that contain `.jpg` images and `.xml` annotations, wherever they are.

In [5]:
def find_dirs(root):
    root = Path(root)
    img_dirs, xml_dirs = set(), set()
    for p in root.rglob('*'):
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
            img_dirs.add(p.parent)
        elif p.suffix.lower() == '.xml':
            xml_dirs.add(p.parent)
    return sorted(img_dirs), sorted(xml_dirs)

img_dirs, xml_dirs = find_dirs(RAW_DIR)
print('Image folders found:')
for d in img_dirs: print('  ', d, '->', len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png'))), 'images')
print('Annotation folders found:')
for d in xml_dirs: print('  ', d, '->', len(list(d.glob('*.xml'))), 'xml')

Image folders found:
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\crazing -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\inclusion -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\patches -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\pitted_surface -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\rolled-in_scale -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\train\images\scratches -> 240 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\validation\images\crazing -> 60 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\validation\images\inclusion -> 60 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\validation\images\patches -> 60 images
   c:\Users\student\Desktop\SteelDefectDetection\data\NEU-DET\validation\

## 4. Index annotations & apply the paper's 8:1:1 split
We follow the paper's protocol (*Sci. Reports* 2025, s41598-025-93469-5): **8:1:1 = 1440 train / 180 val / 180 test**.

NEU-DET ships an official split — `train/` (1440 = 240/class) and `validation/` (360 = 60/class). The official `train/` already equals the paper's train count, so it becomes **train** unchanged. The 360-image `validation/` pool is split **deterministically (seed=42), stratified per class**, into **30/class val + 30/class test**. In Ultralytics terms, `val` (180) is the model-selection set used during training; `test` (180) is held out for the final reported mAP — the paper's headline number.

In [6]:
import random
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']

# Index every annotation by filename stem (robust to misplaced .xml files).
annots = {p.stem: p for p in RAW_DIR.rglob('*.xml')}

def split_image_root(raw, split):
    """Find the '<split>/images' folder, e.g. train/images or validation/images."""
    for p in raw.rglob('images'):
        if p.is_dir() and p.parent.name == split:
            return p
    return None

def collect(raw_split):
    """Return [(stem, img_path, xml_path), ...] for one raw split, plus any images missing an xml."""
    root = split_image_root(RAW_DIR, raw_split)
    assert root is not None, f'Could not find {raw_split}/images under {RAW_DIR}'
    out, missing = [], []
    for img in sorted(root.rglob('*')):
        if img.suffix.lower() not in IMG_EXT:
            continue
        xml = annots.get(img.stem)
        if xml is None:
            missing.append(img.name); continue
        out.append((img.stem, img, xml))
    return out, missing

# Paper's 8:1:1 split (s41598-025-93469-5):
#   official train/ (1440, 240/class)      -> train  (kept as-is)
#   official validation/ (360, 60/class)   -> 30/class val + 30/class test
# Deterministic (seed=42), stratified per class. This MUST match src/make_paper_split.py:
# same CLASSES order, same per-class sort, single Random(42) shuffled across classes in order.
train_pairs, miss_tr = collect('train')
valpool_pairs, miss_va = collect('validation')
missing = miss_tr + miss_va

N_VAL = N_TEST = 30
rng = random.Random(42)
val_pairs, test_pairs = [], []
for c in CLASSES:
    items = sorted([p for p in valpool_pairs if p[0].startswith(c + '_')])  # sort by stem
    rng.shuffle(items)
    val_pairs  += items[:N_VAL]
    test_pairs += items[N_VAL:N_VAL + N_TEST]
    assert len(items) == N_VAL + N_TEST, f'{c}: expected 60 in val pool, got {len(items)}'

split_pairs = {'train': train_pairs, 'val': val_pairs, 'test': test_pairs}
print(f"{len(annots)} annotations indexed")
print(f"train: {len(train_pairs)}   val: {len(val_pairs)}   test: {len(test_pairs)}   (paper 8:1:1)")
if missing:
    print(f'WARNING: {len(missing)} images without annotation:', missing[:10])

1800 annotations indexed
train: 1440   val: 180   test: 180   (paper 8:1:1)


## 5. VOC → YOLO conversion
YOLO label line: `class_id  x_center  y_center  width  height` (all normalized).

In [7]:
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
ALIASES = {'cr':'crazing','in':'inclusion','pa':'patches','ps':'pitted_surface',
           'rs':'rolled-in_scale','sc':'scratches'}

def norm_name(n):
    n = n.strip().lower().replace(' ', '_')
    return ALIASES.get(n, n)

def xml_to_yolo_lines(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    W = float(size.find('width').text); H = float(size.find('height').text)
    lines = []
    for obj in root.iter('object'):
        name = norm_name(obj.find('name').text)
        if name not in CLASSES:
            continue
        cid = CLASSES.index(name)
        b = obj.find('bndbox')
        xmin=float(b.find('xmin').text); xmax=float(b.find('xmax').text)
        ymin=float(b.find('ymin').text); ymax=float(b.find('ymax').text)
        xc=((xmin+xmax)/2)/W; yc=((ymin+ymax)/2)/H
        bw=(xmax-xmin)/W; bh=(ymax-ymin)/H
        lines.append(f'{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

print('Conversion function ready.')

Conversion function ready.


## 6. Build the YOLO dataset
Copy each image into `images/{train,val}/` and write its converted label into
`labels/{train,val}/`, following the official split from step 4.

In [8]:
import shutil
OUT = DATA / 'neu-det-yolo'

# Clean any prior split so a re-run can't leave stale files (e.g. switching 2-way -> 3-way).
for sub in ['images/train','images/val','images/test','labels/train','labels/val','labels/test']:
    d = OUT / sub
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

def materialize(pairs, split_name):
    n_boxes = 0
    for stem, img_path, xml_path in pairs:
        shutil.copy(img_path, OUT / f'images/{split_name}' / (stem + img_path.suffix))
        lines = xml_to_yolo_lines(xml_path)
        (OUT / f'labels/{split_name}' / (stem + '.txt')).write_text('\n'.join(lines))
        n_boxes += len(lines)
    return n_boxes

tb = materialize(split_pairs['train'], 'train')
vb = materialize(split_pairs['val'], 'val')
sb = materialize(split_pairs['test'], 'test')
print(f'Wrote {tb} train boxes, {vb} val boxes, {sb} test boxes')
print('Dataset ready at:', OUT)

Wrote 3335 train boxes, 441 val boxes, 413 test boxes
Dataset ready at: c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo


## 7. Write the YOLO data config
This `data.yaml` is what you pass to `model.train(data=...)`.

In [ ]:
yaml_text = f'''path: {OUT.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 6
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches
'''
cfg_path = OUT / 'data.yaml'
cfg_path.write_text(yaml_text)
print('Wrote', cfg_path)
print(yaml_text)

Wrote c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml
path: c:/Users/student/Desktop/SteelDefectDetection/data/neu-det-yolo
train: images/train
val: images/val
test: images/test

nc: 6
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches



: 

✅ **Data preparation complete.** Next: open `02_eda.ipynb`.